# Trabajo Práctico Final — Clasificación de Emociones
## Paso 5: Interpretabilidad — SHAP Values + Grad-CAM

**¿Por qué importa la interpretabilidad?**
Un modelo que sólo entrega una etiqueta es una "caja negra". Para confiar en él (y para que la cátedra pueda evaluarlo correctamente), necesitamos entender:
- ¿Qué características del vector latente impulsan cada predicción? → **SHAP Values** sobre XGBoost
- ¿Qué regiones del rostro activan la CNN? → **Grad-CAM** sobre MiniEmotionNet
- ¿Qué patrones de frecuencia o textura distinguen cada emoción? → Análisis combinado

---

## 5.0 — Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

try:
    import shap
    SHAP_OK = True
except ImportError:
    print('SHAP no instalado — pip install shap')
    SHAP_OK = False

try:
    from xgboost import XGBClassifier
    import xgboost as xgb
    XGBOOST_OK = True
except ImportError:
    XGBOOST_OK = False

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_DIR = Path(r'C:\Users\Uriel\Documents\UTN\PIYS')
CLASSES  = ['angry','disgusted','fearful','happy','neutral','sad','surprised']
N_CLS    = len(CLASSES)

EMOTION_COLORS = {
    'angry':'#E74C3C','disgusted':'#8E44AD','fearful':'#2980B9',
    'happy':'#F1C40F','neutral':'#95A5A6','sad':'#2C3E50','surprised':'#E67E22'
}

print(f'Dispositivo: {DEVICE}')
print(f'SHAP: {SHAP_OK} | XGBoost: {XGBOOST_OK}')

## 5.1 — Reconstruir modelos del Paso 2

In [ ]:
# ── MiniEmotionNet ──────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if pool: layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class MiniEmotionNet(nn.Module):
    def __init__(self, n_classes=7, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1,   32, pool=False),
            ConvBlock(32,  64,  pool=True),
            ConvBlock(64,  128, pool=True),
            ConvBlock(128, 256, pool=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(128, n_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)
    def extract_features(self, x):
        return self.pool(self.features(x)).flatten(1)


model_cnn = MiniEmotionNet(N_CLS).to(DEVICE)
model_cnn.load_state_dict(torch.load(BASE_DIR / 'best_mini_emotion_net.pt', map_location=DEVICE))
model_cnn.eval()
print('MiniEmotionNet cargada.')

# ── ResNet18 ─────────────────────────────────────────────────────────────────
def build_resnet18(n_classes=7):
    m = models.resnet18(weights=None)
    m.fc = nn.Sequential(
        nn.Dropout(0.4), nn.Linear(512, 256), nn.ReLU(inplace=True),
        nn.Dropout(0.3), nn.Linear(256, n_classes),
    )
    return m

model_resnet = build_resnet18(N_CLS).to(DEVICE)
model_resnet.load_state_dict(torch.load(BASE_DIR / 'best_resnet18_emotion.pt', map_location=DEVICE))
model_resnet.eval()
print('ResNet18 cargada.')

## 5.2 — SHAP Values sobre XGBoost

**SHAP (SHapley Additive exPlanations)** asigna a cada feature una contribución marginal a la predicción, basándose en la teoría de juegos cooperativos. Para cada muestra, el valor SHAP de la feature $i$ indica:
- **Positivo:** la feature empuja la predicción hacia la clase en cuestión
- **Negativo:** la feature aleja la predicción de esa clase
- **La suma de todos los SHAP + valor base = log-odds de la predicción**

In [ ]:
if SHAP_OK and XGBOOST_OK:
    # Reentrenar XGBoost o cargarlo si fue guardado
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    X_train_cnn = np.load(BASE_DIR / 'feat_train_cnn.npy')
    y_train_cnn = np.load(BASE_DIR / 'feat_train_labels_cnn.npy')
    X_test_cnn  = np.load(BASE_DIR / 'feat_test_cnn.npy')
    y_test_cnn  = np.load(BASE_DIR / 'feat_test_labels_cnn.npy')

    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train_cnn)
    X_te_sc = scaler.transform(X_test_cnn)

    xgb_model = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=SEED, n_jobs=-1
    )
    Xtr, Xval, ytr, yval = train_test_split(X_tr_sc, y_train_cnn, test_size=0.15,
                                             stratify=y_train_cnn, random_state=SEED)
    xgb_model.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
    print('XGBoost listo para SHAP.')

    # Explicador de árbol (muy eficiente)
    explainer = shap.TreeExplainer(xgb_model)

    # SHAP sobre una muestra del test para velocidad
    N_SHAP = min(500, len(X_te_sc))
    idx    = np.random.choice(len(X_te_sc), N_SHAP, replace=False)
    X_shap = X_te_sc[idx]
    y_shap = y_test_cnn[idx]

    shap_values = explainer.shap_values(X_shap)  # lista de 7 matrices (una por clase)
    print(f'SHAP values shape: {np.array(shap_values).shape}  (clases × muestras × features)')

In [ ]:
if SHAP_OK and XGBOOST_OK:
    feat_names = [f'f{i}' for i in range(X_shap.shape[1])]

    # Summary plot global (todas las clases)
    fig, axes = plt.subplots(2, 4, figsize=(20, 12))
    axes = axes.flatten()

    for cls_idx, cls in enumerate(CLASSES):
        ax = axes[cls_idx]
        shap_cls = shap_values[cls_idx]  # (N_SHAP, n_features)

        # Top 15 features por importancia SHAP media
        mean_abs_shap = np.abs(shap_cls).mean(axis=0)
        top15 = np.argsort(mean_abs_shap)[::-1][:15]

        y_pos = range(15)
        colors_bar = [EMOTION_COLORS[cls]] * 15
        ax.barh(list(y_pos), mean_abs_shap[top15][::-1],
                color=colors_bar, alpha=0.85, edgecolor='white')
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels([f'feat_{i}' for i in top15[::-1]], fontsize=7)
        ax.set_title(f'{cls.upper()}', fontweight='bold', color=EMOTION_COLORS[cls])
        ax.set_xlabel('|SHAP| medio', fontsize=8)
        ax.grid(axis='x', alpha=0.3)

    axes[-1].axis('off')
    plt.suptitle('SHAP Values — Top 15 features por clase\n'
                 '|SHAP| alto = la feature influye fuertemente en predecir esa emoción',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'fig_20_shap_per_class.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if SHAP_OK and XGBOOST_OK:
    # Beeswarm plot para la clase 'happy' y 'fearful' (usando shap nativo)
    for cls_name, cls_idx in [('happy', 3), ('fearful', 2)]:
        explanation = shap.Explanation(
            values=shap_values[cls_idx],
            base_values=explainer.expected_value[cls_idx] * np.ones(N_SHAP),
            data=X_shap,
            feature_names=feat_names
        )
        fig, ax = plt.subplots(figsize=(10, 6))
        shap.plots.beeswarm(explanation, max_display=20, show=False)
        plt.title(f'SHAP Beeswarm — clase "{cls_name.upper()}"\n'
                  f'Rojo = valor alto de feature | Azul = valor bajo',
                  fontweight='bold')
        plt.tight_layout()
        plt.savefig(BASE_DIR / f'fig_21_shap_beeswarm_{cls_name}.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
if SHAP_OK and XGBOOST_OK:
    # Heatmap de SHAP: features vs clases (importancia media absoluta)
    # Seleccionamos las 20 features con mayor importancia total
    total_importance = np.sum([np.abs(shap_values[i]).mean(axis=0) for i in range(N_CLS)], axis=0)
    top20_global = np.argsort(total_importance)[::-1][:20]

    shap_matrix = np.array([
        np.abs(shap_values[i]).mean(axis=0)[top20_global]
        for i in range(N_CLS)
    ])  # (7, 20)

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(
        shap_matrix,
        xticklabels=[f'f{i}' for i in top20_global],
        yticklabels=CLASSES,
        cmap='YlOrRd', ax=ax,
        annot=True, fmt='.3f', annot_kws={'size': 7},
        cbar_kws={'label': '|SHAP| medio'}
    )
    ax.set_title('Mapa de calor SHAP — ¿qué features importan para cada emoción?\n'
                 '(Top 20 features globalmente más importantes)',
                 fontweight='bold')
    ax.set_xlabel('Feature del encoder CNN')
    ax.set_ylabel('Emoción')
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'fig_22_shap_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5.3 — Grad-CAM sobre MiniEmotionNet

**Grad-CAM (Gradient-weighted Class Activation Mapping):** usa los gradientes de la clase objetivo fluyendo hacia la última capa convolucional para producir un mapa de calor que indica qué regiones del rostro fueron más importantes para la decisión.

Matemáticamente:
$$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A^k_{ij}}$$
$$L^c_{\text{Grad-CAM}} = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right)$$

Donde $A^k$ es el mapa de activación del k-ésimo filtro de la última capa convolucional.

In [ ]:
class GradCAM:
    """
    Implementación de Grad-CAM para cualquier modelo PyTorch.
    Se engancha a la capa target con hooks de forward y backward.
    """
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, target_class=None):
        """
        input_tensor: (1, C, H, W) con requires_grad=True
        target_class: índice de la clase objetivo (None = clase predicha)
        """
        self.model.zero_grad()
        output = self.model(input_tensor)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # Backprop de la clase objetivo
        score = output[0, target_class]
        score.backward()

        # Pesos = media global de los gradientes sobre H×W
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)

        # Combinación lineal ponderada de los mapas de activación
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H', W')
        cam = F.relu(cam)

        # Normalizar a [0,1] y redimensionar a tamaño de imagen original
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        probs = F.softmax(output, dim=1).squeeze().cpu().detach().numpy()
        return cam, target_class, probs


def overlay_gradcam(img_np, cam, alpha=0.5, colormap=cv2.COLORMAP_JET):
    """Superpone el mapa Grad-CAM sobre la imagen original."""
    H, W = img_np.shape[:2]
    cam_resized = cv2.resize(cam, (W, H))
    heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), colormap)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    if img_np.max() <= 1.0:
        img_rgb = (img_np * 255).astype(np.uint8)
    else:
        img_rgb = img_np.astype(np.uint8)

    if len(img_rgb.shape) == 2:
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_GRAY2RGB)

    overlay = (alpha * heatmap + (1 - alpha) * img_rgb).astype(np.uint8)
    return overlay, heatmap


# Instanciar Grad-CAM sobre el último bloque convolucional
target_layer = model_cnn.features[-1].block[-3]  # última Conv2d antes del MaxPool
gradcam = GradCAM(model_cnn, target_layer)

stats = pd.read_csv(BASE_DIR / 'normalization_stats.csv')
NORM_MEAN = float(stats['mean'].values[0])
NORM_STD  = float(stats['std'].values[0])

print('Grad-CAM instanciado sobre la capa:', target_layer)

In [ ]:
import albumentations as A

aug_eval = A.Compose([A.Normalize(mean=[NORM_MEAN], std=[NORM_STD])])

df_test = pd.read_csv(BASE_DIR / 'df_test_split.csv')

def load_and_preprocess(path):
    img_raw = np.array(Image.open(path).convert('L'), dtype=np.uint8)
    img_norm = aug_eval(image=img_raw)['image'].astype(np.float32)
    tensor = torch.tensor(img_norm).unsqueeze(0).unsqueeze(0).to(DEVICE)  # (1,1,48,48)
    tensor.requires_grad_(True)
    return img_raw, tensor


# Grad-CAM: 2 muestras por clase, mostrando predicción correcta e incorrecta
fig, axes = plt.subplots(N_CLS, 5, figsize=(18, N_CLS * 2.5))

col_titles = ['Imagen original', 'Mapa Grad-CAM', 'Overlay', 'Imagen original (2)', 'Overlay (2)']

for row, cls in enumerate(CLASSES):
    cls_idx = CLASSES.index(cls)
    subset  = df_test[df_test['label'] == cls].sample(min(2, len(df_test[df_test['label']==cls])),
                                                       random_state=row)

    sample_col = 0
    for _, rec in subset.iterrows():
        img_raw, tensor = load_and_preprocess(rec['path'])
        cam, pred_cls, probs = gradcam.generate(tensor, target_class=cls_idx)
        overlay, heatmap = overlay_gradcam(img_raw, cam)

        base_col = sample_col * 3
        if base_col >= 5:
            break

        conf = probs[cls_idx]
        pred_lbl = CLASSES[pred_cls]
        correct = '✓' if pred_cls == cls_idx else '✗'

        axes[row, base_col].imshow(img_raw, cmap='gray')
        axes[row, base_col].set_title(f'{correct} Pred: {pred_lbl}\n({conf:.2%})', fontsize=7)
        axes[row, base_col].axis('off')

        if base_col + 1 < 5:
            axes[row, base_col+1].imshow(heatmap)
            axes[row, base_col+1].set_title('Grad-CAM', fontsize=7)
            axes[row, base_col+1].axis('off')

        if base_col + 2 < 5:
            axes[row, base_col+2].imshow(overlay)
            axes[row, base_col+2].set_title('Overlay', fontsize=7)
            axes[row, base_col+2].axis('off')

        sample_col += 1

    # Etiqueta de la fila
    axes[row, 0].set_ylabel(cls.upper(), fontsize=9, fontweight='bold',
                            color=EMOTION_COLORS[cls], rotation=0, ha='right',
                            labelpad=60, va='center')

    # Apagar celdas sin usar
    for c in range(sample_col*3, 5):
        axes[row, c].axis('off')

plt.suptitle('Grad-CAM — Regiones del rostro que activan la CNN por emoción\n'
             'Zona roja = máxima activación | Zona azul = mínima activación',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_23_gradcam_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.4 — Grad-CAM: predicciones correctas vs incorrectas

Comparamos el Grad-CAM de predicciones **correctas** (el modelo mira las zonas correctas del rostro) vs **incorrectas** (el modelo se distrae con zonas irrelevantes).

In [ ]:
y_true_all = np.load(BASE_DIR / 'cnn_y_true.npy')
y_pred_all = np.load(BASE_DIR / 'cnn_y_pred.npy')

# Añadir índice original del test al dataframe
df_test_eval = df_test.copy().reset_index(drop=True)
df_test_eval['y_pred'] = y_pred_all[:len(df_test_eval)]
df_test_eval['correct'] = (df_test_eval['label_enc'] == df_test_eval['y_pred'])

fig, axes = plt.subplots(4, 6, figsize=(18, 13))
row_ctr = 0

# Para cada emoción: 1 correcto + 1 incorrecto
for cls in ['fearful', 'surprised', 'angry', 'sad']:
    cls_idx = CLASSES.index(cls)
    correct_sample   = df_test_eval[(df_test_eval['label']==cls) & df_test_eval['correct']].head(1)
    incorrect_sample = df_test_eval[(df_test_eval['label']==cls) & ~df_test_eval['correct']].head(1)

    for status, sample in [('CORRECTO', correct_sample), ('INCORRECTO', incorrect_sample)]:
        if len(sample) == 0:
            for c in range(3):
                axes[row_ctr, c].axis('off')
            row_ctr += 1
            continue

        rec = sample.iloc[0]
        img_raw, tensor = load_and_preprocess(rec['path'])
        cam, pred_cls, probs = gradcam.generate(tensor, target_class=cls_idx)
        overlay, heatmap = overlay_gradcam(img_raw, cam)

        pred_label = CLASSES[pred_cls]
        conf = probs[pred_cls]
        color = 'green' if status == 'CORRECTO' else 'red'

        axes[row_ctr, 0].imshow(img_raw, cmap='gray')
        axes[row_ctr, 0].set_title(f'{cls.upper()} [{status}]\nPredicción: {pred_label} ({conf:.1%})',
                                   fontsize=8, color=color, fontweight='bold')
        axes[row_ctr, 0].axis('off')

        axes[row_ctr, 1].imshow(heatmap)
        axes[row_ctr, 1].set_title('Grad-CAM', fontsize=8)
        axes[row_ctr, 1].axis('off')

        axes[row_ctr, 2].imshow(overlay)
        axes[row_ctr, 2].set_title('Overlay', fontsize=8)
        axes[row_ctr, 2].axis('off')

        row_ctr += 1

# Apagar celdas del lado derecho (mostramos 3 de 6 columnas por bloque, repetimos para 2 bloques)
for r in range(4):
    for c in range(3, 6):
        axes[r, c].axis('off')

plt.suptitle('Grad-CAM — Predicciones CORRECTAS vs INCORRECTAS\n'
             'En errores: el mapa de calor suele enfocarse en zonas faciales equivocadas',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_24_gradcam_correct_vs_wrong.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.5 — Grad-CAM promedio por clase

Promediando los mapas Grad-CAM de todas las muestras de una clase, obtenemos el **patrón facial característico** que el modelo aprendió para cada emoción.

In [ ]:
N_SAMPLES_CAM = 30  # muestras por clase para el promedio

fig, axes = plt.subplots(2, N_CLS, figsize=(N_CLS * 3, 7))

for col, cls in enumerate(CLASSES):
    cls_idx = CLASSES.index(cls)
    subset  = df_test[df_test['label'] == cls].sample(
        min(N_SAMPLES_CAM, len(df_test[df_test['label']==cls])),
        random_state=SEED
    )

    cams_all, imgs_all = [], []
    for _, rec in subset.iterrows():
        try:
            img_raw, tensor = load_and_preprocess(rec['path'])
            cam, _, _ = gradcam.generate(tensor, target_class=cls_idx)
            cam_resized = cv2.resize(cam, (48, 48))
            cams_all.append(cam_resized)
            imgs_all.append(img_raw.astype(np.float32) / 255.0)
        except Exception:
            pass

    avg_cam = np.mean(cams_all, axis=0)
    avg_img = np.mean(imgs_all, axis=0)
    overlay_avg, heatmap_avg = overlay_gradcam(avg_img, avg_cam)

    axes[0, col].imshow(avg_img, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(cls.upper(), fontweight='bold',
                           color=EMOTION_COLORS[cls], fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(overlay_avg)
    axes[1, col].set_title(f'n={len(cams_all)}', fontsize=8)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Imagen\npromedio', fontsize=9, rotation=0, ha='right', labelpad=60)
axes[1, 0].set_ylabel('Grad-CAM\npromedio', fontsize=9, rotation=0, ha='right', labelpad=60)

plt.suptitle('Grad-CAM Promedio por Emoción — Patrón facial aprendido por la CNN\n'
             'Zona roja = región más determinante para cada emoción',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_25_gradcam_avg_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.6 — Resumen de Interpretabilidad

### Hallazgos esperados

| Técnica | Emoción | Región/Feature determinante |
|---|---|---|
| Grad-CAM | HAPPY | Zona de la boca y mejillas (sonrisa) |
| Grad-CAM | ANGRY | Cejas fruncidas y entrecejo |
| Grad-CAM | SURPRISED | Ojos abiertos y cejas elevadas |
| Grad-CAM | SAD | Comisuras de la boca hacia abajo, ojos |
| Grad-CAM | FEARFUL | Ojos muy abiertos (similar a SURPRISED) |
| SHAP | Todas | Features latentes del encoder que capturan músculos faciales específicos (Action Units de FACS) |

### Conexión con la materia
> El Grad-CAM usa **gradientes** (derivadas) que fluyen hacia atrás por la red, exactamente como el backpropagation que entrena la red. La operación de promediar los gradientes sobre el mapa espacial equivale a calcular la **correlación** entre los gradientes y las activaciones, concepto central de `Clase_Convol_Correl_PSUTN.pdf`.

## Resumen del Paso 5 — Figuras generadas

| Figura | Descripción |
|---|---|
| `fig_20_shap_per_class.png` | Top 15 features SHAP por emoción (barplot) |
| `fig_21_shap_beeswarm_happy/fearful.png` | Beeswarm plot SHAP detallado |
| `fig_22_shap_heatmap.png` | Mapa de calor SHAP: features × emociones |
| `fig_23_gradcam_per_class.png` | Grad-CAM para 2 muestras de cada emoción |
| `fig_24_gradcam_correct_vs_wrong.png` | Grad-CAM correcto vs incorrecto |
| `fig_25_gradcam_avg_per_class.png` | Grad-CAM promediado — patrón facial por emoción |